<a href="https://colab.research.google.com/github/Dina-Shanjida/IMDB_sentimental_analysis_Simple_RNN/blob/main/IMDB_movie_review_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## GPU setup

In [2]:
import torch
import torch.nn as nn
import pandas as pd
import re
import kagglehub
import os
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
!pip install -q kagglehub

## Load dataset

In [4]:
path = kagglehub.dataset_download(
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
)

df = pd.read_csv(
    os.path.join(path, "IMDB Dataset.csv")
)

df['label'] = df['sentiment'].map({'positive':1, 'negative':0})

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [5]:
df = df.sample(20000, random_state=42).reset_index(drop=True)
df.head()

,review,sentiment,label
0,I really liked this Summerslam due to the look...,positive,1
1,Not many television shows appeal to quite as m...,positive,1
2,The film quickly gets to a major chase scene w...,negative,0
3,Jane Austen would definitely approve of this o...,positive,1
4,Expectations were somewhat high for me when I ...,negative,0


## tokenize

In [6]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

## vocab build

In [7]:
from collections import Counter

counter = Counter()

for text in df['review']:
    counter.update(tokenize(text))

vocab = {"<UNK>": 0}

for word, freq in counter.items():
    if freq > 2:
        vocab[word] = len(vocab)

print("Vocab Size:", len(vocab))

Vocab Size: 35020


## text to indices

In [8]:
MAX_LEN = 200

def text_to_indices(text):
    tokens = tokenize(text)[:MAX_LEN]
    return [vocab.get(t, 0) for t in tokens]

## train test val split

In [9]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

## dataset class

In [10]:
class IMDbDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.iloc[idx]['review']
        label = self.df.iloc[idx]['label']

        x = text_to_indices(text)

        return torch.tensor(x), torch.tensor(label, dtype=torch.float)

## padding

In [11]:
def collate_fn(batch):
    texts = [x[0] for x in batch]
    labels = torch.tensor([x[1] for x in batch])

    texts = pad_sequence(
        texts,
        batch_first=True,
        padding_value=0
    )

    texts = texts[:, :MAX_LEN]

    return texts, labels

## dataloader

In [12]:
train_loader = DataLoader(
    IMDbDataset(train_df),
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    IMDbDataset(val_df),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    IMDbDataset(test_df),
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

## simple RNN model

In [13]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=256):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True, nonlinearity='tanh')
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, h = self.rnn(x)
        out = self.fc(h.squeeze(0))
        return out

In [14]:
model = SimpleRNN(len(vocab)).to(device)


## training

In [15]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        out = model(x).squeeze(1)

        loss = criterion(out, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

Epoch 1 Loss: 307.1334
Epoch 2 Loss: 301.8607
Epoch 3 Loss: 295.2312
Epoch 4 Loss: 287.5023
Epoch 5 Loss: 275.0758
Epoch 6 Loss: 259.8222
Epoch 7 Loss: 241.1613
Epoch 8 Loss: 219.5535
Epoch 9 Loss: 201.8722
Epoch 10 Loss: 190.7324
Epoch 11 Loss: 184.9265
Epoch 12 Loss: 179.9576
Epoch 13 Loss: 170.1501
Epoch 14 Loss: 181.6224
Epoch 15 Loss: 172.0941
Epoch 16 Loss: 169.5264
Epoch 17 Loss: 167.2928
Epoch 18 Loss: 167.4507
Epoch 19 Loss: 175.8343
Epoch 20 Loss: 171.2391
Epoch 21 Loss: 165.9453
Epoch 22 Loss: 163.9668
Epoch 23 Loss: 165.9050
Epoch 24 Loss: 170.5538
Epoch 25 Loss: 164.8553
Epoch 26 Loss: 163.5855
Epoch 27 Loss: 167.1821
Epoch 28 Loss: 171.7134
Epoch 29 Loss: 167.2727
Epoch 30 Loss: 166.0959


## evaluation

In [16]:
def accuracy(loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            out = model(x).squeeze(1)
            preds = torch.sigmoid(out) > 0.5

            correct += (preds == y.bool()).sum().item()
            total += y.size(0)

    return correct / total

In [17]:
print("Train Acc:", accuracy(train_loader))
print("Val Acc:", accuracy(val_loader))
print("Test Acc:", accuracy(test_loader))

Train Acc: 0.7357857142857143
Val Acc: 0.5333333333333333
Test Acc: 0.5236666666666666


## prediction

In [18]:
def predict(text):
    model.eval()

    x = torch.tensor(text_to_indices(text)).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(x)
        prob = torch.sigmoid(out).item()

    return "positive" if prob > 0.5 else "negative"

In [19]:
print(predict("This movie was absolutely amazing"))
print(predict("Worst movie I have ever seen"))

positive
negative
